# dfs-three-set-toposort composite — cx19: DFS toposort with cycle detection via temp/perm/visiting trio

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `dfs-three-set-toposort`, `cycle-detection-temp-set`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "dfs-three-set-toposort"
DD_ATOM_IDS = ["dfs-three-set-toposort", "cycle-detection-temp-set"]
DD_SUBTOPICS = ["Backprop: DFS three-set toposort", "Backprop: cycle detection via temp set"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing DFS toposort with temp-set cycle detection

The two atoms are TWO RESPONSIBILITIES of one three-colour DFS walk:

- **`dfs-three-set-toposort`** — produce a deps-first ordering of every
  reachable node (root LAST). The output is the `result` list.
- **`cycle-detection-temp-set`** — the `temp` (gray) set on the recursion
  stack: re-entering it means we walked a back-edge → cycle.

```python
def topo_sort_with_cycle_check(root, get_children):
    perm = set()     # finished subtrees (cycle-detection-temp-set: NOT a cycle)
    temp = set()     # currently-on-stack (cycle-detection-temp-set: IS a cycle)
    result = []
    def visit(node):
        nid = id(node)
        if nid in perm: return                # legal shared descendant
        if nid in temp:                        # back-edge → CYCLE
            raise ValueError(f'cycle at {node!r}')
        temp.add(nid)
        for child in get_children(node):
            visit(child)
        temp.remove(nid)                       # pop off recursion stack
        perm.add(nid)
        result.append(node)                    # deps-first append
    visit(root)
    return result
```

**Why both atoms compose into one function.** A DAG-only toposort needs
BOTH outputs: a valid ordering AND a guarantee it ran on a DAG. Drop
the temp-set check and you'd silently produce a partial list on a
cyclic input. Drop the perm-set and you'd flag every diamond DAG as a
false cycle. Two color sets, two purposes — both load-bearing.

**Result order: deps-first, root LAST.** This is the order a FORWARD
pass would use (leaves first, then their consumers). The reverse pass
wants the OPPOSITE — `[::-1]` on the output flips it. That's the next
composite (cx22).

### Composite Exercise — DFS toposort with cycle detection via temp/perm/visiting trio

**Atoms exercised together**: `dfs-three-set-toposort`, `cycle-detection-temp-set`

Implement `cx19_topo_sort_with_cycle_check(root, get_children)` — a three-colour DFS that returns descendants of `root` in deps-first order (root LAST) AND raises `ValueError` on any cycle.

**Both atoms in one function.** This is the canonical ARENA `topological_sort` — every later composite (cx20 backprop loop, cx22 sorted-graph builder) uses it.

**Contract.**
- Returns `list` of nodes reachable from `root` via `get_children`.
- Every node appears AFTER all of its transitive children — deps-first.
- `root` is the LAST element.
- Each reachable node appears EXACTLY once (diamond DAGs collapse).
- Raises `ValueError` on a cycle (self-loop, two-node, deep-graph).

**Algorithm** — three colours, both atoms together:

```python
def visit(node):
    nid = id(node)
    if nid in perm: return                  # already finished
    if nid in temp: raise ValueError(...)   # cycle (temp-set atom)
    temp.add(nid)
    for child in get_children(node):
        visit(child)
    temp.remove(nid)                        # pop off stack
    perm.add(nid)
    result.append(node)                     # deps-first (toposort atom)
```

Use `id(node)` as the set key — safe for any object identity.

**Failure modes the test catches.**
1. Forgetting `temp.remove(nid)` → siblings sharing a leaf falsely flag as cycle.
2. Forgetting `perm.add(nid)` → diamond DAGs falsely flag as cycle.
3. Appending BEFORE recursing into children → wrong order (root would be first).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx19_topo_sort_with_cycle_check(root, get_children):
    """Three-colour DFS: deps-first toposort, raise ValueError on cycle."""
    raise NotImplementedError()

def _test_cx19():
    # --- helper graph node ---
    class N:
        def __init__(self, name, *children):
            self.name = name
            self.children = list(children)
        def __repr__(self): return f'N({self.name})'

    def get_children(n): return n.children

    # === linear chain a -> b -> c — deps-first, root last ===
    c = N('c'); b = N('b', c); a = N('a', b)
    order = cx19_topo_sort_with_cycle_check(a, get_children)
    names = [n.name for n in order]
    assert names == ['c', 'b', 'a'], f'linear: {names}'

    # === diamond DAG — d appears ONCE despite two paths ===
    d = N('d'); b = N('b', d); c = N('c', d); a = N('a', b, c)
    order = cx19_topo_sort_with_cycle_check(a, get_children)
    names = [n.name for n in order]
    assert names.count('d') == 1, f'd must appear once: {names}'
    assert names[-1] == 'a', f'root LAST: {names}'
    assert names.index('d') < names.index('b') < names.index('a')
    assert names.index('d') < names.index('c') < names.index('a')

    # === self-loop → ValueError (cycle-detection-temp-set in action) ===
    s = N('s'); s.children = [s]
    raised = False
    try: cx19_topo_sort_with_cycle_check(s, get_children)
    except ValueError: raised = True
    assert raised, 'self-loop must raise ValueError'

    # === two-node cycle → ValueError ===
    x = N('x'); y = N('y')
    x.children = [y]; y.children = [x]
    raised = False
    try: cx19_topo_sort_with_cycle_check(x, get_children)
    except ValueError: raised = True
    assert raised, 'two-node cycle must raise ValueError'

    # === mid-graph cycle (p -> q -> r -> q) → ValueError ===
    p = N('p'); q = N('q'); r = N('r')
    p.children = [q]; q.children = [r]; r.children = [q]
    raised = False
    try: cx19_topo_sort_with_cycle_check(p, get_children)
    except ValueError: raised = True
    assert raised, 'mid-graph cycle must raise'

    # === Two siblings share a leaf — NOT a cycle (regression: temp.remove) ===
    leaf = N('leaf'); xx = N('x', leaf); yy = N('y', leaf)
    root = N('root', xx, yy)
    order = cx19_topo_sort_with_cycle_check(root, get_children)
    names = [n.name for n in order]
    assert names.count('leaf') == 1, 'shared leaf must appear ONCE'
    assert names[-1] == 'root', 'root LAST despite shared descendant'

    # === singleton — single node, no children ===
    lonely = N('lonely')
    order = cx19_topo_sort_with_cycle_check(lonely, get_children)
    assert order == [lonely], f'singleton: {order}'
    _dd_passed.add('cx19')

_test_cx19()

<details><summary>Show solution — cx19</summary>

```python
def cx19_topo_sort_with_cycle_check(root, get_children):
    result = []
    perm = set()   # fully processed (NOT a cycle if re-visited)
    temp = set()   # currently on DFS stack (IS a cycle if re-visited)

    def visit(node):
        nid = id(node)
        if nid in perm:
            return                          # legal shared descendant
        if nid in temp:
            raise ValueError(f'cycle at {node!r}')
        temp.add(nid)
        for child in get_children(node):
            visit(child)
        temp.remove(nid)                    # pop off recursion stack
        perm.add(nid)
        result.append(node)                 # deps-first append

    visit(root)
    return result
```

**Two color sets, two atoms, two failure modes.** `perm` is the `cycle-detection-temp-set` insight that 'finished subtree' is not the same as 'currently in-flight'. `temp` is the back-edge detector. Drop either and the algorithm breaks on either diamond DAGs (false cycle) or shared-leaf siblings (false cycle from forgotten remove).

**`result.append(node)` is the `dfs-three-set-toposort` insight.** The append happens AFTER all children have been visited — that's what makes it deps-first. Appending before recursing would give you root-first order (which is what the reverse pass eventually wants, but the standard idiom builds deps-first and reverses).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx19'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx19',
        'subtopics': ["Backprop: DFS three-set toposort", "Backprop: cycle detection via temp set"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()